In [1]:
# 1D free rotor: test of winding truncation error scaling
# Copy-paste this whole cell into Colab and run.

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

# Use double precision
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# Exact rotor partition function Z_exact(β) at θ = 0
# -----------------------------
def Z_exact(beta, N_n=80):
    """
    Exact canonical partition function for the free rotor with θ=0,
    using truncated sum over n ∈ ℤ: H = 0.5 * n^2 (ħ=I=1).
    """
    n = np.arange(-N_n, N_n + 1, dtype=np.float64)
    energies = 0.5 * n**2
    weights = np.exp(-beta * energies)
    return weights.sum()

# -----------------------------
# Tensor-network rotor partition function Z_TN(β; M, K_max)
# -----------------------------
def Z_TN_free(beta, M, K_max, device=device):
    """
    Sign-problem-free TN approximation for the free rotor partition function
    using unwrapped Gaussian kernel on the covering space.

    Parameters:
      beta   : inverse temperature
      M      : number of angle grid points on the circle
      K_max  : max winding sheet index (integer)
    """
    # Angle grid on [0, 2π)
    phi = torch.linspace(0.0, 2.0 * math.pi, M + 1, device=device)[:-1]  # shape (M,)
    dphi = 2.0 * math.pi / M

    # Sheets k = -K_max,...,K_max
    ks = torch.arange(-K_max, K_max + 1, device=device)  # shape (n_sheets,)
    n_sheets = ks.shape[0]

    # Extended coordinates x_{(k,a)} = φ_a + 2π k
    two_pi_k = (2.0 * math.pi * ks).view(n_sheets, 1).repeat(1, M)      # (n_sheets, M)
    phi_grid = phi.view(1, M).repeat(n_sheets, 1)                        # (n_sheets, M)
    x = (phi_grid + two_pi_k).reshape(-1)                                # (D,), D = M * n_sheets

    # Build W_ij ≈ K(x_i, x_j; β) * dφ
    x_i = x.view(-1, 1)  # (D,1)
    x_j = x.view(1, -1)  # (1,D)
    dist_sq = (x_i - x_j) ** 2

    pref = math.sqrt(1.0 / (2.0 * math.pi * beta))
    W = pref * torch.exp(-dist_sq / (2.0 * beta)) * dphi  # (D,D), real, non-negative

    # Indices for sheet k=0 (start sheet) and sheet k (target sheet)
    k_idx0 = K_max
    start_angles = torch.arange(M, device=device)
    start_indices = k_idx0 * M + start_angles

    Z_TN = torch.zeros((), dtype=torch.float64, device=device)

    # θ = 0 ⇒ phase = 1 for all k
    for k_val in range(-K_max, K_max + 1):
        k_idx = k_val + K_max
        target_indices = k_idx * M + start_angles

        block = W[target_indices][:, start_indices]  # shape (M,M)
        trace_k = torch.sum(torch.diagonal(block))
        Z_TN = Z_TN + trace_k

    return float(Z_TN.cpu().numpy())

# -----------------------------
# Experiment: error vs K_max, test Gaussian scaling
# -----------------------------
betas = [0.5, 1.0, 2.0]   # inverse temperatures to test
M = 256                   # angle grid size (large enough for good spatial accuracy)
K_values = list(range(0, 7))  # K_max = 0..6

for beta in betas:
    print("\n==============================")
    print(f"β = {beta}")
    Zex = Z_exact(beta)
    print(f"Z_exact(β) ≈ {Zex:.16e}")

    errors = []
    for K in K_values:
        Ztn = Z_TN_free(beta, M, K)
        err = abs(Ztn - Zex)
        errors.append(err)
        print(f"  K_max={K:2d}:  Z_TN={Ztn: .16e},  |ΔZ|={err:.3e}")

    # Prepare data for linear fit: log(error) vs K_max^2
    xs = []
    ys = []
    for K, err in zip(K_values, errors):
        if K > 0 and err > 1e-14:  # avoid log(0) and K=0
            xs.append(K**2)
            ys.append(math.log(err))

    if len(xs) >= 2:
        xs_np = np.array(xs)
        ys_np = np.array(ys)
        A = np.vstack([xs_np, np.ones_like(xs_np)]).T
        slope, intercept = np.linalg.lstsq(A, ys_np, rcond=None)[0]
        predicted = -2.0 * math.pi**2 / beta
        print(f"Fitted slope d log(|ΔZ|)/d(K_max^2) ≈ {slope:.3f},  predicted ≈ {predicted:.3f}")

        # Plot log(error) vs K^2 for this β
        plt.figure()
        plt.scatter(xs_np, ys_np, label="data")
        plt.plot(xs_np, slope*xs_np + intercept, label="fit")
        plt.xlabel("$K_{\\max}^2$")
        plt.ylabel("$\\log |Z_{TN} - Z_{exact}|$")
        plt.title(f"β = {beta}, slope ≈ {slope:.3f}, pred ≈ {predicted:.3f}")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("Not enough nonzero error points for a reliable fit at this β.")


Using device: cuda

β = 0.5
Z_exact(β) ≈ 3.5449077018110318e+00
  K_max= 0:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 1:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 2:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 3:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 4:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 5:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
  K_max= 6:  Z_TN= 3.5449077018110318e+00,  |ΔZ|=0.000e+00
Not enough nonzero error points for a reliable fit at this β.

β = 1.0
Z_exact(β) ≈ 2.5066282880429056e+00
  K_max= 0:  Z_TN= 2.5066282746310007e+00,  |ΔZ|=1.341e-08
  K_max= 1:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
  K_max= 2:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
  K_max= 3:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
  K_max= 4:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
  K_max= 5:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
  K_max= 6:  Z_TN= 2.5066282880429060e+00,  |ΔZ|=4.441e-16
Not